In [2]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

### 🕵️‍♂️ 구글 뉴스 묶음(LU3Rqb) 구조 분석

* **묶음 단위**: `<div class="LU3Rqb">` 
    * 이 태그 하나가 뉴스 4개(메인 1 + 관련 3)를 감싸는 부모입니다.
* **기사 제목**: `<a class="gPFEn">`
    * 이 클래스가 실제 뉴스 제목 텍스트를 가지고 있습니다.
* **추출 전략**:
    1. `LU3Rqb`를 모두 리스트로 받는다.
    2. 각 묶음(bundle) 안에서 `gPFEn`을 다시 찾는다.
    3. 첫 번째 `gPFEn`을 해당 묶음의 대표 키워드로 활용한다.

In [ ]:
# 1. 주제별 큰 덩어리들을 먼저 찾습니다.
# 구글 뉴스의 경우 각 주제가 <section>이나 특정 div로 묶여 있습니다.
topic_sections = driver.find_elements(By.CSS_SELECTOR, 'section[class^="..."]') 

all_news_data = []

for section in topic_sections:
    # 각 주제 상자 안에서 '주제 이름'을 먼저 뽑습니다.
    topic_name = section.find_element(By.CSS_SELECTOR, 'h2').text
    
    # 2. 해당 주제 상자 안에서 기사 4개를 찾습니다.
    # section.find_elements를 쓰면 '그 상자 안에서만' 찾게 됩니다.
    articles = section.find_elements(By.CSS_SELECTOR, 'article')
    
    for article in articles:
        title = article.find_element(By.CSS_SELECTOR, 'h3').text
        link = article.find_element(By.CSS_SELECTOR, 'a').get_attribute('href')
        
        # 데이터를 주제별로 묶어서 저장
        all_news_data.append({
            "주제": topic_name,
            "제목": title,
            "링크": link
        })

In [3]:
# 1. 브라우저 설정
chrome_options = Options()
# chrome_options.add_argument('--headless') # 필요할 때 주석 해제

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

all_news = []
print("🚀 구글 세계 뉴스 크롤링 시작!")

try:
    url = 'https://news.google.com/topics/CAAqJggKIiBDQkFTRWdvSUwyMHZNRGx1YlY4U0FtdHZHZ0pMVWlnQVAB?hl=ko&gl=KR&ceid=KR%3Ako'
    driver.get(url)
    time.sleep(3) # 페이지 로딩 대기

    # 1. 뉴스 묶음(바구니)들을 모두 가져옵니다.
    bundles = driver.find_elements(By.CSS_SELECTOR, 'div.LU3Rqb')
    
    # [종료 조건]: 수집할 바구니가 없으면 종료
    if not bundles:
        print("수집할 뉴스 묶음이 없습니다.")
    else:
        for bundle in bundles:
            try:
                print("="*50)
                # 2. 이 묶음의 '대표 제목' (로그용)
                main_title_el = bundle.find_element(By.CSS_SELECTOR, 'a.gPFEn')
                print(f"[주제 그룹]: {main_title_el.text[:30]}...")
                
                # 3. 이 묶음 안에 있는 모든 기사 요소들을 찾습니다.
                # 각 기사 영역은 보통 div.IBr9hb(메인) 또는 div.f9uzM(서브)입니다.
                # 여기서는 기사 정보를 담고 있는 개별 컨테이너를 잡는 게 안전합니다.
                articles = bundle.find_elements(By.CSS_SELECTOR, 'div[jslog]') # 기사 단위 컨테이너
                
                for article_item in articles:
                    try:
                        # article_item(개별 기사 박스) 안에서 정보를 찾습니다.
                        author = article_item.find_element(By.CSS_SELECTOR, '.vr1PYe').text
                        title = article_item.find_element(By.CSS_SELECTOR, 'a.gPFEn').text
                        # 시간 정보 (클래스 점 오타 수정: .hvbAAd)
                        when = article_item.find_element(By.CSS_SELECTOR, '.hvbAAd').text

                        all_news.append({
                            'Title': title,
                            'Author': author,
                            'Time': when,
                            'Main_Topic': main_title_el.text # 어떤 묶음인지 표시
                        })
                    except:
                        continue # 기사 하나 에러나도 다음 기사로
                        
            except Exception as e:
                print(f"묶음 처리 중 스킵: {e}")
                continue
        
        print(f"📦 수집 완료: {len(all_news)}개 기사")
        
except Exception as e:
    print(f"❌ 에러 발생: {e}")

finally:
    driver.quit()
    print(f"✨ 총 {len(all_news)}개의 데이터를 수집했습니다.")

# 2. 데이터 저장 및 확인
if all_news:
    df = pd.DataFrame(all_news)
    df.to_csv('news_All.csv', index=False, encoding='utf-8-sig')
    print("✅ 파일 저장 완료!")
else:
    print("데이터가 없어 저장하지 않았습니다.")

🚀 구글 세계 뉴스 크롤링 시작!
[주제 그룹]: 관세 ‘만능키’ 꺼낸 미국, 무역법 301조 조사 개시...
[주제 그룹]: “전쟁 안 밀린다” 이란, 트럼프 특사 휴전 요청 두 ...
[주제 그룹]: [속보] 장동혁 "지방선거까지 윤리위 추가 징계 중단 ...
[주제 그룹]: 호르무즈해협서 태국·일본 선박 등 4척 피격···이란 ...
[주제 그룹]: ‘오래된 지도로 잘못 공격’…미군, 이란 초교 ‘170...
[주제 그룹]: “슈퍼마리오 실사판” 日도심서 대형 파이프 도로에 솟아...
[주제 그룹]: “막 때리지 말고 눈치 좀”…美, 이스라엘에 이란 에너...
[주제 그룹]: 네타냐후, 전쟁 지속하려 국방예산 18조 원 추가편성 ...
[주제 그룹]: “장관 이상하게 나왔다” 美국방부, 사진기자 출입 금지...
[주제 그룹]: 김정은, 김주애 대동 전략순항미사일 시험발사 참관...
[주제 그룹]: "이란, 美 캘리포니아에 드론 기습 공격 의도…FBI ...
[주제 그룹]: [美 이란 공습] 중동 전쟁에 원유 우려 고조된 亞…휴...
[주제 그룹]: 한국인 11명, 사우디서 일본 전세기 타고 도쿄 도착...
[주제 그룹]: 미군, 이란전 부상자 수 첫 공식 발표…10일간 140...
[주제 그룹]: '가성비템' 천궁-2 수요 증가‥외신도 K-방산 주목...
[주제 그룹]: 年 100발도 못만드는 토마호크, 수백발 쏟아부어…美 ...
📦 수집 완료: 80개 기사
✨ 총 80개의 데이터를 수집했습니다.
✅ 파일 저장 완료!


In [4]:
df

,Title,Author,Time,Main_Topic
0,"관세 ‘만능키’ 꺼낸 미국, 무역법 301조 조사 개시…한국 포함",한겨레,3시간 전,"관세 ‘만능키’ 꺼낸 미국, 무역법 301조 조사 개시…한국 포함"
1,"美, 전쟁 중에도 ‘관세 전쟁’... “한국 車·기계·철강 과잉생산”",조선일보,5시간 전,"관세 ‘만능키’ 꺼낸 미국, 무역법 301조 조사 개시…한국 포함"
2,"美, 전쟁 중에도 ‘관세 전쟁’... “한국 車·기계·철강 과잉생산”",조선일보,5시간 전,"관세 ‘만능키’ 꺼낸 미국, 무역법 301조 조사 개시…한국 포함"
3,"美 301조 조사에 ""한미간 협상 이행에 최선""…정부 숨은 의도는?",뉴시스,37분 전,"관세 ‘만능키’ 꺼낸 미국, 무역법 301조 조사 개시…한국 포함"
4,"청와대 ""주요국 대비 불리하지 않도록 미 측과 적극 협의""",YTN 사이언스,5시간 전,"관세 ‘만능키’ 꺼낸 미국, 무역법 301조 조사 개시…한국 포함"
...,...,...,...,...
75,"年 100발도 못만드는 토마호크, 수백발 쏟아부어…美 무기부족 우려",동아일보,19시간 전,"年 100발도 못만드는 토마호크, 수백발 쏟아부어…美 무기부족 우려"
76,"美, 이란전 6일간 16조 썼다... 탄약 소모도 예상보다 빨라",조선일보,7시간 전,"年 100발도 못만드는 토마호크, 수백발 쏟아부어…美 무기부족 우려"
77,"美, 이란전 6일간 16조 썼다... 탄약 소모도 예상보다 빨라",조선일보,7시간 전,"年 100발도 못만드는 토마호크, 수백발 쏟아부어…美 무기부족 우려"
78,"미국, 이란 공격에 매일 2조8000억원 썼다···미 국방부, 의회에 비용 추정치 보고",경향신문,3시간 전,"年 100발도 못만드는 토마호크, 수백발 쏟아부어…美 무기부족 우려"
